# Free-Living Analysis with Clinical Metadata (UPDRS / Hoehn & Yahr)
Processes `parkinson_trunk_cwa_matches.csv`: downloads each trunk recording (`device_position == "Back"`), runs gait analysis, and combines the result with each patient's clinical severity (UPDRS, HY).

**Applied decisions:**
- The `device_position` column is trusted (the 6 cases where the file name does not match are not excluded).
- Patients with multiple trunk sessions are **averaged** into a single final row.
- The patient identifier used throughout the analysis is `patient_key` -- `patient_name` is discarded from the beginning (it is not stored in any output CSV).

The total size is ~8.5 GB -- the script downloads and deletes each `.cwa` one at a time and can be paused/resumed (completed sessions are not reprocessed).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q scikit-digital-health requests


In [ ]:
import os
import sys
import pandas as pd

sys.path.append('/content/drive/MyDrive/Lasige')  # carpeta donde esta gait_lib.py
from gait_lib import build_gait_old, run_gait_freeliving, download_file


## 2. Configuration

In [ ]:
MATCHES_CSV = "/content/drive/MyDrive/Lasige/parkinson_trunk_cwa_matches.csv"

DOWNLOAD_DIR = "/content/freeliving_cwa"       # Colab local disk, NOT Drive (faster; deleted when finished)
OUTPUT_DIR   = "/content/drive/MyDrive/Lasige/FreeLiving_Matched"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

PER_SESSION_CSV = os.path.join(OUTPUT_DIR, "Per_session_freeliving_gait.csv")
PER_PATIENT_CSV = os.path.join(OUTPUT_DIR, "Per_patient_freeliving_gait.csv")

MAX_SESSIONS = 5          # test limit; increase the number once you confirm it works
DELETE_AFTER_PROCESSING = True   # borra el .cwa tras analizarlo (recomendado, son ~150-270MB cada uno)
PATIENT_HEIGHT_DEFAULT_M = 1.69  # used if 'height' in the file is not a valid number


## 3. Load and clean the manifest

Filters only `device_position == "Back"` and immediately discards `patient_name` (real names are not wanted in any output CSV).

In [ ]:
manifest = pd.read_csv(MATCHES_CSV)

trunk_manifest = manifest[manifest["device_position"] == "Back"].copy()
trunk_manifest = trunk_manifest.drop(columns=["patient_name"], errors="ignore")

print(f"Trunk sessions (Back): {len(trunk_manifest)}")
print(f"Unique patients: {trunk_manifest['patient_key'].nunique()}")

trunk_manifest[["patient_key", "patient_code", "filename", "mdsupdrs_total", "hoehn_and_yahr"]].head(10)


## 4. Download + analyze each session (resumable)

If `Per_session_freeliving_gait.csv` already exists, sessions that already appear there are skipped -- you can stop the cell and run it again later without losing progress.

In [ ]:
gait = build_gait_old()

if os.path.exists(PER_SESSION_CSV):
    session_results = pd.read_csv(PER_SESSION_CSV)
    already_done = set(session_results["filename"])
    print(f"Retomando: {len(already_done)} sessions ya procesadas anteriormente")
else:
    session_results = pd.DataFrame()
    already_done = set()

new_rows = []
processed_now = 0

for _, row in trunk_manifest.iterrows():
    if processed_now >= MAX_SESSIONS:
        break
    if row["filename"] in already_done:
        continue

    print(f"\n[{row['patient_key']} / {row['filename']}] descargando ({row['file_size_mb']:.0f} MB)...")
    local_path = os.path.join(DOWNLOAD_DIR, row["filename"])

    try:
        download_file(row["signed_download_url"], local_path)

        height_m = row["height"] / 100 if pd.notna(row["height"]) and row["height"] < 3 else (
            row["height"] / 100 if pd.notna(row["height"]) else PATIENT_HEIGHT_DEFAULT_M
        )

        summary_df = run_gait_freeliving(
            gait_model=gait,
            patient_id=row["patient_key"],
            file_path=local_path,
            output_dir=OUTPUT_DIR,
            group="PD",
            height_m=height_m,
            fs=None,
        )

        if summary_df is not None:
            summary_df["filename"] = row["filename"]
            summary_df["patient_code"] = row["patient_code"]
            summary_df["mdsupdrs_total"] = row["mdsupdrs_total"]
            summary_df["mdsupdrs_part_III"] = row["mdsupdrs_part_III"]
            summary_df["hoehn_and_yahr"] = row["hoehn_and_yahr"]
            new_rows.append(summary_df)
            processed_now += 1

    except Exception as e:
        print(f"[{row['filename']}] ERROR: {e}")

    finally:
        if DELETE_AFTER_PROCESSING and os.path.exists(local_path):
            os.remove(local_path)

if new_rows:
    session_results = pd.concat([session_results] + new_rows, ignore_index=True)
    session_results.to_csv(PER_SESSION_CSV, index=False)
    print(f"\nSaved -> {PER_SESSION_CSV} ({len(session_results)} sessions en total)")
else:
    print("\nNo new session was processed (or all sessions were already completed).")


Run Cell 4 as many times as needed (increasing `MAX_SESSIONS` or leaving it fixed) until no pending sessions remain.

## 5. Validate sessions before averaging

Two checks:
1. **Impossible duration** (<=0h or >500h) -- corrupted timestamps inside the `.cwa` (e.g., a clock reset halfway through the recording). These sessions are **excluded** from the per-patient average; if the patient has another valid session, only that session remains in the analysis.
2. **Suspiciously short duration** (< `MIN_RECORDING_HOURS`) -- the recording is not necessarily corrupted, but a few hours are not comparable with several days of free-living recording in the rest of the group. These are **not automatically excluded** (it could be a short but valid recording) -- they are marked with `short_recording_flag=True` so you can decide whether to remove them before the final analysis.

In [ ]:
MIN_RECORDING_HOURS = 24  # threshold for flagging (not excluding) sessions as "short"

# 1) impossible duration -> excluded from the average
invalid_duration = session_results[
    (session_results["recording_hours"] <= 0) | (session_results["recording_hours"] > 500)
]
if len(invalid_duration) > 0:
    print(f"[EXCLUIDAS] {len(invalid_duration)} session(s) with invalid duration:")
    print(invalid_duration[["patient_id", "filename", "recording_hours"]].to_string(index=False))

session_results = session_results[
    (session_results["recording_hours"] > 0) & (session_results["recording_hours"] <= 500)
].reset_index(drop=True)

# 2) short duration -> flagged, not excluded
session_results["short_recording_flag"] = session_results["recording_hours"] < MIN_RECORDING_HOURS

short_sessions = session_results[session_results["short_recording_flag"]]
if len(short_sessions) > 0:
    print(f"\n[AVISO] {len(short_sessions)} session(s) with less than {MIN_RECORDING_HOURS}h "
          f"(marcadas, no excluidas):")
    print(short_sessions[["patient_id", "filename", "recording_hours", "n_steps"]].to_string(index=False))

session_results.to_csv(PER_SESSION_CSV, index=False)


## 6. Average sessions per patient

A patient with multiple sessions (baseline/discharge) remains in **a single row**, averaging the numeric columns. `n_sessions` indicates how many sessions were averaged. `short_recording_flag` is also averaged -- if it is > 0, at least one of the patient's sessions was short.

In [ ]:
numeric_cols = session_results.select_dtypes(include="number").columns.tolist()

per_patient = (
    session_results
    .groupby("patient_id")[numeric_cols]
    .mean()
    .reset_index()
    .rename(columns={"patient_id": "patient_key"})
)

n_sessions = session_results.groupby("patient_id").size().rename("n_sessions")
per_patient = per_patient.merge(n_sessions, left_on="patient_key", right_index=True)

# patient_code is not numeric -- take the first one (it is the same patient in all rows)
codes = session_results.groupby("patient_id")["patient_code"].first()
per_patient = per_patient.merge(codes, left_on="patient_key", right_index=True)

per_patient.to_csv(PER_PATIENT_CSV, index=False)
print(f"Saved -> {PER_PATIENT_CSV} ({len(per_patient)} patients)")

per_patient


## 7. Combine the bout-duration breakdown (all patients)

`run_gait_freeliving` already saves, for each session, a CSV with metrics broken down into `5-15s / 15-30s / 30-60s / >60s` categories. This cell combines them into a single CSV to analyze short vs. long bout patterns across the whole group.

In [ ]:
import glob

bout_cat_files = glob.glob(os.path.join(OUTPUT_DIR, "*_freeliving_bouts_by_duration.csv"))

if bout_cat_files:
    all_bout_categories = pd.concat(
        [pd.read_csv(f) for f in bout_cat_files], ignore_index=True
    )
    all_bout_categories_path = os.path.join(OUTPUT_DIR, "ALL_patients_freeliving_bouts_by_duration.csv")
    all_bout_categories.to_csv(all_bout_categories_path, index=False)
    print(f"Saved -> {all_bout_categories_path} ({len(bout_cat_files)} sessions combined)")
else:
    print("There are not yet any *_freeliving_bouts_by_duration.csv -- run first la Cell 4.")

all_bout_categories.head(10) if bout_cat_files else None


## 8. Correlation with clinical severity

Now that each patient has their UPDRS/HY data, we can examine whether any free-living gait metric is related to motor severity.

In [ ]:
import matplotlib.pyplot as plt

X_METRIC = "mdsupdrs_part_III"   # subescala motora del UPDRS
Y_METRIC = "PARAM:cadence_mean"  # change to steps_per_day, PARAM:gait speed_mean, etc.

valid = per_patient[[X_METRIC, Y_METRIC]].dropna()
r = valid.corr().iloc[0, 1] if len(valid) > 2 else float("nan")

plt.figure(figsize=(6, 5))
plt.scatter(valid[X_METRIC], valid[Y_METRIC], alpha=0.8)
plt.xlabel(X_METRIC)
plt.ylabel(Y_METRIC)
plt.title(f"r = {r:.2f}  (n={len(valid)})")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "FreeLiving_severity_correlation.png"), dpi=200)
plt.show()
